# 09 - Pipeline `two-stream` (RGB + frequency fusion)

The integrative contribution: a **spatial RGB CNN** in parallel with a **frequency CNN** (on the per-image log-magnitude FFT), fused into a final decision. Generated images deviate from real ones in the high-frequency band (see EDA §5) — the frequency stream targets exactly that. **Multi-component:** we report a `p_fake` per stream **and** the fused output.

**Sections:** 0 Setup · 1 Data · 2 Model · 3 Training setup · 4 Train (visible loop, fused + per-stream aux loss) · 5 Curves · 6 In-distribution eval (+ components) · 7 Cross-generator (OOD) preview · 8 Grad-CAM · 9 Save metrics.json.

Requires `03_split_and_preprocessing`. Artifacts → `notebooks/artifacts/two-stream/{models,figures,metrics}`.

## 0 - Setup

In [ ]:
import sys, time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display

_here = Path.cwd()
_nb_dir = _here if (_here / "utils").is_dir() else _here / "notebooks"
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

from utils import datasets as D, models as M, training as T, metrics as Me, viz as V, explain as E, eda
from utils.paths import repo_paths, artifact_dirs

torch.manual_seed(42); np.random.seed(42)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATHS = repo_paths(_nb_dir)
DATA_DIR = PATHS["data"]; AIR_DIR = DATA_DIR / "ai-real-images"
SPLIT_PATH = AIR_DIR / "manifest_split.csv"
TINY_MANIFEST = DATA_DIR / "tiny-genimage" / "manifest_clean.csv"

PIPELINE = "two-stream"
WORKING_SIZE = 128
NORM = "dataset"
BATCH_SIZE = 128
EPOCHS = 30
WARMUP_EPOCHS = 3
LABEL_SMOOTH = 0.05
AUX_WEIGHT = 0.5      # weight on each per-stream auxiliary loss
NUM_WORKERS = 8
dirs = artifact_dirs(PIPELINE)
print("device:", device, "| pipeline:", PIPELINE)

## 1 - Data

128px / dataset-norm loaders (same as the from-scratch CNNs). The frequency input is derived inside the model, so no extra data plumbing is needed.

In [ ]:
loaders = D.make_loaders(SPLIT_PATH, working_size=WORKING_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, norm=NORM)
train_loader, val_loader, test_loader = loaders["train"], loaders["val"], loaders["test"]
mean, std = D.resolve_stats(NORM, AIR_DIR)
split_df = pd.read_csv(SPLIT_PATH); split_df = split_df[split_df["keep"]]
test_df = split_df[split_df["split_final"] == "test"].reset_index(drop=True)
print(f"train {len(train_loader.dataset):,} | val {len(val_loader.dataset):,} | test {len(test_loader.dataset):,}")

## 2 - Model

Two `_FeatCNN` branches: spatial (3-channel RGB) and frequency (1-channel log-magnitude FFT of luminance, computed on the fly in float32). Features concatenate → fusion head. Each branch also has an auxiliary head so we get a per-stream `p_fake`.

In [ ]:
model = M.build_two_stream(feat=256, p_drop=0.3).to(device, memory_format=torch.channels_last)
print("params:", f"{M.count_params(model):,}")
with torch.no_grad():
    fused, s, q = model.forward_all(torch.randn(2, 3, WORKING_SIZE, WORKING_SIZE, device=device))
print("forward_all shapes:", tuple(fused.shape), tuple(s.shape), tuple(q.shape))

## 3 - Training setup

AdamW + cosine warmup; joint training of both branches + fusion. Total loss = fused BCE + 0.5·(spatial BCE + frequency BCE).

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
spe = len(train_loader)
scheduler = T.build_cosine_with_warmup(optimizer, total_steps=EPOCHS * spe, warmup_steps=WARMUP_EPOCHS * spe)
stopper = T.EarlyStopper(mode="max", patience=7, min_delta=1e-3)
history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_acc": []}
best_auc = -1.0; ckpt_path = dirs["models"] / "best.pt"

## 4 - Train (visible loop, fused + auxiliary losses)

Custom loop because the model has three outputs. Validation uses the fused output (`model(x)` → fused) via the shared `evaluate`.

In [ ]:
for epoch in range(EPOCHS):
    t0 = time.time(); model.train(); running, n = 0.0, 0
    for x, y in train_loader:
        x = x.to(device, memory_format=torch.channels_last, non_blocking=True)
        y = y.to(device, non_blocking=True).float()
        ys = T.smooth_binary_targets(y, LABEL_SMOOTH)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            fused, s, q = model.forward_all(x)
            loss = loss_fn(fused, ys) + AUX_WEIGHT * (loss_fn(s, ys) + loss_fn(q, ys))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad(set_to_none=True); scheduler.step()
        running += loss.item() * y.size(0); n += y.size(0)
    train_loss = running / n
    yv, pv, vloss = T.evaluate(model, val_loader, device, loss_fn)   # fused output
    vm = Me.classification_metrics(yv, pv)
    history["train_loss"].append(train_loss); history["val_loss"].append(vloss)
    history["val_auc"].append(vm["auc_roc"]); history["val_acc"].append(vm["accuracy"])
    improved, stop = stopper.step(vm["auc_roc"])
    if improved:
        best_auc = vm["auc_roc"]
        T.save_checkpoint(ckpt_path, model, optimizer, epoch=epoch, best_metric=best_auc, extra={"history": history})
    print(f"epoch {epoch+1:02d} | train_loss {train_loss:.4f} | val_loss {vloss:.4f} | val_auc {vm['auc_roc']:.4f} | val_acc {vm['accuracy']:.4f} | {time.time()-t0:.0f}s{'  *best' if improved else ''}")
    if stop:
        print("early stopping"); break

## 5 - Training curves

In [ ]:
V.plot_training_curves(history).savefig(dirs["figures"] / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6 - In-distribution evaluation (+ per-component)

Fused metrics at 0.5 and val-tuned threshold, plus per-stream (spatial / frequency) metrics so we can see how much each branch contributes.

In [ ]:
@torch.no_grad()
def eval_components(model, loader, device):
    model.eval(); P = {"fused": [], "spatial": [], "frequency": []}; Y = []
    for x, y in loader:
        x = x.to(device, memory_format=torch.channels_last, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            fused, s, q = model.forward_all(x)
        P["fused"].append(torch.sigmoid(fused).float().cpu().numpy())
        P["spatial"].append(torch.sigmoid(s).float().cpu().numpy())
        P["frequency"].append(torch.sigmoid(q).float().cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(Y).astype(int), {k: np.concatenate(v) for k, v in P.items()}

T.load_checkpoint(ckpt_path, model, map_location=device)
yt, P = eval_components(model, test_loader, device)
yv, pv, _ = T.evaluate(model, val_loader, device)
tuned = Me.best_f1_threshold(yv, pv)
m05 = Me.classification_metrics(yt, P["fused"], threshold=0.5)
mtuned = Me.classification_metrics(yt, P["fused"], threshold=tuned["threshold"])
comp = {"spatial": Me.classification_metrics(yt, P["spatial"]), "frequency": Me.classification_metrics(yt, P["frequency"])}
print("tuned threshold (on val):", round(tuned["threshold"], 4))
print(f"fused acc {m05['accuracy']:.4f} auc {m05['auc_roc']:.4f} | spatial acc {comp['spatial']['accuracy']:.4f} auc {comp['spatial']['auc_roc']:.4f} | freq acc {comp['frequency']['accuracy']:.4f} auc {comp['frequency']['auc_roc']:.4f}")
display(Me.summary_table(m05))
V.plot_confusion(m05["confusion_matrix"]).savefig(dirs["figures"] / "confusion.png", dpi=150, bbox_inches="tight")
V.plot_roc_pr(yt, P["fused"]).savefig(dirs["figures"] / "roc_pr.png", dpi=150, bbox_inches="tight")
V.plot_reliability(yt, P["fused"]).savefig(dirs["figures"] / "reliability.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 - Cross-generator (OOD) preview

In [ ]:
GEN_MAP = {
    "imagenet_ai_0419_biggan": "biggan", "imagenet_ai_0419_vqdm": "vqdm",
    "imagenet_ai_0424_sdv5": "sdv5", "imagenet_ai_0424_wukong": "wukong",
    "imagenet_ai_0508_adm": "adm", "imagenet_glide": "glide", "imagenet_midjourney": "midjourney",
}
ood_loader, ood_df = D.make_ood_loader(TINY_MANIFEST, WORKING_SIZE, BATCH_SIZE, mean, std, num_workers=NUM_WORKERS)
yo, po, _ = T.evaluate(model, ood_loader, device)   # fused
ood_df = ood_df.assign(p_fake=po, y_true=yo)
ood_df["y_pred"] = (ood_df["p_fake"] >= 0.5).astype(int)
ood_df["generator"] = ood_df["source"].map(GEN_MAP).fillna(ood_df["source"])
rows = [{"generator": gen, "accuracy": float((g["y_pred"] == g["y_true"]).mean()), "n": int(len(g))}
        for gen, g in ood_df.groupby("generator")]
per_gen = pd.DataFrame(rows)
overall_ood = float((ood_df["y_pred"] == ood_df["y_true"]).mean())
display(per_gen)
print(f"overall OOD accuracy: {overall_ood:.4f}  (in-dist test acc: {m05['accuracy']:.4f})")
V.plot_per_generator_bar(per_gen, ref_acc=m05["accuracy"]).savefig(dirs["figures"] / "ood_per_generator.png", dpi=150, bbox_inches="tight")
plt.show()

## 8 - Explainability (Grad-CAM on the spatial stream)

Grad-CAM w.r.t. the fused decision, localized on the spatial branch's last conv block.

In [ ]:
eval_tf = D.build_eval_tf(WORKING_SIZE, mean, std)
target_layers = [model.spatial.features[-1]]
examples = E.pick_examples(test_df, n_per_class=3, seed=0)
fig, axes = plt.subplots(2, len(examples), figsize=(2.2 * len(examples), 4.6))
model.eval()
for j, ex in enumerate(examples):
    arr = eda.read_rgb(ex["filepath"])
    xt = eval_tf(torch.from_numpy(arr).permute(2, 0, 1))
    x = xt.unsqueeze(0).to(device)
    rgb = D.denormalize(xt, mean, std).permute(1, 2, 0).numpy()
    with torch.no_grad():
        p = torch.sigmoid(model(x)).item()
    overlay = E.gradcam_overlay(model, target_layers, x, rgb)
    axes[0, j].imshow(rgb); axes[0, j].axis("off"); axes[0, j].set_title(f"{ex['label']}  p_fake={p:.2f}", fontsize=8)
    axes[1, j].imshow(overlay); axes[1, j].axis("off")
fig.suptitle("two-stream Grad-CAM (spatial branch)", fontsize=11)
fig.savefig(dirs["figures"] / "gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 - Save metrics.json

Includes the fused in-distribution metrics plus a `components` block (spatial / frequency), matching the multi-component prediction schema the app uses.

In [ ]:
record = {
    "pipeline": PIPELINE,
    "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "working_size": WORKING_SIZE, "normalization": NORM,
    "dataset": {"in_distribution": "ai-real-images", "ood": "tiny-genimage"},
    "threshold_default": 0.5, "threshold_tuned": tuned["threshold"],
    "in_distribution": {"at_0.5": m05, "at_tuned": mtuned},
    "components": {"spatial": comp["spatial"], "frequency": comp["frequency"]},
    "ood": {"overall_accuracy": overall_ood,
             "per_generator": {r.generator: {"accuracy": r.accuracy, "n": r.n} for r in per_gen.itertuples()},
             "preview": True},
    "train_history": {"epochs": len(history["val_auc"]), "best_epoch": int(np.argmax(history["val_auc"])) + 1, "best_val_auc": float(max(history["val_auc"]))},
    "figures": {k: f"figures/{k}.png" for k in ["training_curves", "confusion", "roc_pr", "reliability", "ood_per_generator", "gradcam"]},
}
Me.save_metrics(record, dirs["metrics"] / "metrics.json")
print("saved", dirs["metrics"] / "metrics.json")
print("\nfused in-dist @0.5:  acc {accuracy:.4f}  f1 {f1_macro:.4f}  auc {auc_roc:.4f}  brier {brier:.4f}".format(**m05))

**That completes the six pipelines.** Next come the comparison/evaluation notebooks (10/11) and the app in `src/`.